In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd

# webdriver_manager를 사용하여 Chrome 드라이버 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# 검색할 구 목록
districts = [
    "서울 종로구", "서울 중구", "서울 용산구", "서울 성동구", "서울 광진구", 
    "서울 동대문구", "서울 중랑구", "서울 성북구", "서울 강북구", "서울 도봉구", 
    "서울 노원구", "서울 은평구", "서울 서대문구", "서울 마포구", "서울 양천구", 
    "서울 강서구", "서울 구로구", "서울 금천구", "서울 영등포구", "서울 동작구", 
    "서울 관악구", "서울 서초구", "서울 강남구", "서울 송파구", "서울 강동구",
    
    "경기도 수원시", "경기도 고양시", "경기도 용인시", "경기도 성남시", "경기도 부천시",
    "경기도 안산시", "경기도 안양시", "경기도 남양주시", "경기도 화성시", "경기도 평택시",
    "경기도 의정부시", "경기도 시흥시", "경기도 파주시", "경기도 김포시", "경기도 광명시",
    "경기도 광주시", "경기도 군포시", "경기도 이천시", "경기도 양주시", "경기도 오산시",
    "경기도 구리시", "경기도 안성시", "경기도 포천시", "경기도 의왕시", "경기도 하남시",
    "경기도 여주시", "경기도 양평군", "경기도 동두천시", "경기도 과천시", "경기도 가평군","경기도 연천군",
    
    "강원도", "경상북도", "경상남도", 
    "충청북도", "충청남도", "전라남도","전라북도",
    "제주도", "부산광역시", "대구광역시",
    "인천광역시", "광주광역시", "대전광역시", "울산광역시",
    "세종특별자치시"
]


# 결과를 저장할 리스트 생성
data = []

for district in districts:
    # 카카오 지도 웹사이트 열기
    driver.get("https://map.kakao.com/")
    
    # 요소가 나타날 때까지 기다린 후 클릭
    time.sleep(2)
    try:
        img = driver.find_element(By.CSS_SELECTOR, "div.inner_coach_layer")
        img.click()
    except:
        pass
    
    # 검색 수행
    search = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.query")
    search.send_keys(f"동물병원 {district}")
    search_btn = driver.find_element(By.CSS_SELECTOR, "#search\\.keyword\\.submit")
    search_btn.click()
    
    # 검색 결과가 로드될 시간을 줌
    time.sleep(2)
    
    # BeautifulSoup을 사용하여 페이지 소스 파싱
    soup = bs(driver.page_source, "lxml")
    
    # 광고 제거
    ads = soup.select("span.ico_ad")
    for ad in ads:
        ad.extract()
    
    # "장소 더보기" 버튼 클릭
    try:
        btn_more = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.place\\.more")
        btn_more.click()
        time.sleep(2)
    except:
        pass
    
    try:
        while True:
            # 1부터 5까지 페이지를 반복
            for page_num in range(1, 6):
                try:
                    page_button = driver.find_element(By.CSS_SELECTOR, f"#info\\.search\\.page\\.no{page_num}")
                    page_button.click()
                    time.sleep(2)
                    
                    soup = bs(driver.page_source, "lxml")
                    
                    # 광고 제거
                    ads = soup.select("span.ico_ad")
                    for ad in ads:
                        ad.extract()
                    
                    # 장소 이름과 도로명 주소 추출
                    titles = soup.select("a.link_name")
                    addresses = soup.select("p[data-id='address']")
                    
                    for title, address in zip(titles, addresses):
                        data.append({
                            "장소": title['title'],
                            "도로명 주소": address['title'],
                            "지역": district
                        })
                except:
                    break
            
            # 다음 버튼 클릭
            next_button = driver.find_element(By.CSS_SELECTOR, "#info\\.search\\.page\\.next")
            if "disabled" in next_button.get_attribute("class"):
                break
            next_button.click()
            time.sleep(2)
    except Exception as e:
        print(f"오류 발생: {e}")

# 드라이버 닫기
driver.quit()

# 데이터를 DataFrame으로 변환
df = pd.DataFrame(data)

# DataFrame을 CSV 파일로 저장
df.to_csv("animal_hospital.csv", index=False, encoding='utf-8-sig')

print("데이터가 'animal_hospital.csv' 파일에 저장되었습니다.")
